# Notebook 02: Document Chunking & Text Preparation
This notebook implements text chunking for Gold Layer business and review documents using windowed character segmentation (`CHUNK_SIZE=1000`, `OVERLAP=200`).

In [1]:
import duckdb
import pandas as pd
from pathlib import Path
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

GOLD_BIZ_PATH = PROJECT_ROOT / 'data' / '02_gold' / 'business_documents'
GOLD_REV_PATH = PROJECT_ROOT / 'data' / '02_gold' / 'review_documents'
CHUNKS_DIR = PROJECT_ROOT / 'data' / '03_chunks'
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 1000
OVERLAP = 200
STEP = CHUNK_SIZE - OVERLAP

print(f'Chunk Configuration:')
print(f'  - Chunk Size: {CHUNK_SIZE} characters')
print(f'  - Overlap:    {OVERLAP} characters')
print(f'  - Step Size:  {STEP} characters')


Chunk Configuration:
  - Chunk Size: 1000 characters
  - Overlap:    200 characters
  - Step Size:  800 characters

In [2]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    if not text or not isinstance(text, str):
        return []
    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(text), step):
        chunks.append(text[i:i + chunk_size])
        if i + chunk_size >= len(text):
            break
    return chunks

# Test chunking on sample document
sample_doc = "Yelp RAG Document Chunking Test. " * 50
sample_chunks = chunk_text(sample_doc, chunk_size=100, overlap=20)
print(f"Sample Input Length:  {len(sample_doc)} chars")
print(f"Total Chunks Created: {len(sample_chunks)}")
print(f"Chunk #0: {sample_chunks[0][:60]}...")
print(f"Chunk #1: {sample_chunks[1][:60]}...")


Sample Input Length:  1650 chars
Total Chunks Created: 21
Chunk #0: Yelp RAG Document Chunking Test. Yelp RAG Document Chunking ...
Chunk #1: ent Chunking Test. Yelp RAG Document Chunking Test. Yelp RAG...

In [3]:
con = duckdb.connect()
biz_glob = str(GOLD_BIZ_PATH / "*.parquet").replace("\\", "/")

biz_df = con.execute(f"SELECT document_id, business_name, city, state, primary_category, business_rating, document_text FROM read_parquet('{biz_glob}') LIMIT 2000").df()

biz_chunk_records = []
for idx, row in biz_df.iterrows():
    doc_id = row["document_id"]
    text = row["document_text"]
    chunks = chunk_text(text)
    for c_idx, chunk in enumerate(chunks):
        chunk_id = f"{doc_id}_{c_idx}"
        biz_chunk_records.append({
            "chunk_id": chunk_id,
            "document_id": doc_id,
            "chunk_number": c_idx,
            "business_name": row["business_name"],
            "city": row["city"],
            "state": row["state"],
            "primary_category": row["primary_category"],
            "business_rating": row["business_rating"],
            "chunk_text": chunk,
            "document_type": "business"
        })

biz_chunks_df = pd.DataFrame(biz_chunk_records)
print("=======================================================")
print("           BUSINESS DOCUMENT CHUNKING METRICS         ")
print("=======================================================")
print(f"Total Documents Processed: {len(biz_df):,}")
print(f"Total Chunks Generated:    {len(biz_chunks_df):,}")
print(f"Avg Chunks per Document:   {len(biz_chunks_df)/len(biz_df):.2f}")
display(biz_chunks_df.head(5))


           BUSINESS DOCUMENT CHUNKING METRICS         
Total Documents Processed: 2,000
Total Chunks Generated:    2,000
Avg Chunks per Document:   1.00
                        chunk_id                    document_id  chunk_number          business_name         city state       primary_category  business_rating                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [4]:
rev_glob = str(GOLD_REV_PATH / "*.parquet").replace("\\", "/")

rev_df = con.execute(f"SELECT document_id, review_id, business_id, business_name, city, state, stars, sentiment, document_text FROM read_parquet('{rev_glob}') LIMIT 2000").df()

rev_chunk_records = []
for idx, row in rev_df.iterrows():
    doc_id = row["document_id"]
    text = row["document_text"]
    chunks = chunk_text(text)
    for c_idx, chunk in enumerate(chunks):
        chunk_id = f"{doc_id}_{c_idx}"
        rev_chunk_records.append({
            "chunk_id": chunk_id,
            "document_id": doc_id,
            "chunk_number": c_idx,
            "business_name": row["business_name"],
            "city": row["city"],
            "state": row["state"],
            "stars": row["stars"],
            "sentiment": row["sentiment"],
            "chunk_text": chunk,
            "document_type": "review"
        })

rev_chunks_df = pd.DataFrame(rev_chunk_records)
print("=======================================================")
print("            REVIEW DOCUMENT CHUNKING METRICS          ")
print("=======================================================")
print(f"Total Documents Processed: {len(rev_df):,}")
print(f"Total Chunks Generated:    {len(rev_chunks_df):,}")
print(f"Avg Chunks per Document:   {len(rev_chunks_df)/len(rev_df):.2f}")
display(rev_chunks_df.head(5))
con.close()


            REVIEW DOCUMENT CHUNKING METRICS          
Total Documents Processed: 2,000
Total Chunks Generated:    2,524
Avg Chunks per Document:   1.26
                        chunk_id                    document_id  chunk_number                                        business_name          city state  stars sentiment                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        